In [4]:
import json
import pandas as pd
import folium
import branca.colormap as cm
import re
import unicodedata

# 1️⃣ Đọc dữ liệu GeoJSON (bản đồ hành chính Việt Nam)
geojson_path = "vietnam_provinces.geojson"
with open(geojson_path, encoding="utf-8") as f:
    geojson_data = json.load(f)

# 2️⃣ Đọc dữ liệu thống kê IP theo AS
file_path = "statistics_qscanner_enriched.csv"
df = pd.read_csv(file_path, on_bad_lines="skip", encoding="utf-8")

# 3️⃣ Lọc dữ liệu hợp lệ
df_valid = df.dropna(subset=['as', 'ip', 'latitude', 'longitude', 'region_name'])

# 4️⃣ Chuẩn hóa tên tỉnh để khớp với `Name_EN` trong GeoJSON
def normalize_province_name(name):
    """ Chuẩn hóa tên tỉnh: xóa dấu, sửa lỗi tên """
    name = str(name).strip().lower()
    name = re.sub(r'province', '', name)
    name = re.sub(r'–', '-', name)
    name = re.sub(r'\s+', ' ', name)
    name_ascii = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = name.title().strip()
    name_ascii = name_ascii.title().strip()
    name_corrections = {
        "Ho Chi Minh": "TP. Ho Chi Minh",
        "Ba Ria-Vung Tau": "Ba Ria - Vung Tau",
        "Haiphong": "Hai Phong",
        "Hanoi": "Ha Noi",
        "Lang Son": "Lang Son",
        "Quang Nam": "Quang Nam",
        "Ninh Binh": "Ninh Binh",
        "Bac Ninh": "Bac Ninh",
        "Son La": "Son La",
        "Tay Ninh": "Tay Ninh",
        "Nghe An": "Nghe An",
        "Ninh Thuan": "Ninh Thuan",
        "Dong Nai": "Dong Nai",
        "Lam Dong": "Lam Dong",
        "Phu Tho": "Phu Tho",
        "Binh Thuan": "Binh Thuan",
        "Quang Ngai": "Quang Ngai",
        "Phu Yen": "Phu Yen",
        "Thua Thien Hue": "Thua Thien - Hue",
        "Ha Tinh": "Ha Tinh",
        "Quang Ninh": "Quang Ninh",
        "Binh Duong": "Binh Duong",
        "Vinh Phuc": "Vinh Phuc",
        "Bac Giang": "Bac Giang",
        "Long An Povince": "Long An",
        "Lâm Đồng": "Lam Dong",
        "Đồng Nai": "Dong Nai"
    }
    return name_corrections.get(name, name_corrections.get(name_ascii, name))

df_valid['region_name_clean'] = df_valid['region_name'].apply(normalize_province_name)

# 5️⃣ Lấy danh sách tỉnh từ GeoJSON
geojson_provinces_en = {feat['properties']['Name_EN'].strip() for feat in geojson_data['features']}

# 6️⃣ Kiểm tra lại các tỉnh chưa khớp
missing_provinces_en = set(df_valid['region_name_clean']) - geojson_provinces_en
print("⚠️ Các tỉnh vẫn chưa khớp với Name_EN sau khi sửa:", missing_provinces_en)

# 7️⃣ Nhóm số lượng IP theo tỉnh
province_summary = df_valid.groupby('region_name_clean')['ip'].count().reset_index()
province_summary.rename(columns={'ip': 'num_ips'}, inplace=True)
province_summary = province_summary[province_summary['region_name_clean'].isin(geojson_provinces_en)]

# 8️⃣ Nhóm theo AS và tỉnh
as_locations = (
    df_valid.groupby(['as', 'region_name_clean'])
    .agg({'latitude': 'mean', 'longitude': 'mean', 'ip': 'count'})
    .reset_index()
)
as_locations.rename(columns={'ip': 'num_ips'}, inplace=True)

# 🔟 Tạo bản đồ
m = folium.Map(location=[16.0, 108.0], zoom_start=6, tiles="CartoDB positron")
colormap = cm.linear.YlOrRd_09.scale(as_locations['num_ips'].min(), as_locations['num_ips'].max())

folium.Choropleth(
    geo_data=geojson_data,
    name="Choropleth",
    data=province_summary,
    columns=["region_name_clean", "num_ips"],
    key_on="feature.properties.Name_EN",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name="Number of IPs by Province",
).add_to(m)

folium.GeoJson(
    geojson_data,
    tooltip=folium.GeoJsonTooltip(fields=["Name_EN"], aliases=["Province: "]),
).add_to(m)

for _, row in as_locations.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=max(row['num_ips'] / 50, 3),
        color=colormap(row['num_ips']),
        fill=True,
        fill_color=colormap(row['num_ips']),
        fill_opacity=0.6,
        popup=f"AS: {row['as']}<br>Province: {row['region_name_clean']}<br>IPs: {row['num_ips']}"
    ).add_to(m)

colormap.caption = "Number of IPs by AS"
m.add_child(colormap)

html_path = "quic_as_vietnam_admin_map_thermal.html"
m.save(html_path)
print(f"✅ Bản đồ đã được lưu tại: {html_path}")


⚠️ Các tỉnh vẫn chưa khớp với Name_EN sau khi sửa: {'-'}
✅ Bản đồ đã được lưu tại: quic_as_vietnam_admin_map_thermal.html


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# =====================
# Data
# =====================
labels = ['Jan 2025', 'Oct 2025', 'Feb 2026', 'Apr 2026']

ipv4_count = np.array([8337, 10279, 10595, 11498], dtype=float)
ipv6_count = np.array([np.nan, np.nan, 3377, 3381], dtype=float)

# These values are already percentages, e.g., 0.051 means 0.051%
ipv4_percent = np.array([0.051, 0.062, 0.064, 0.070], dtype=float)
ipv6_percent = np.array([np.nan, np.nan, 0.395, 0.395], dtype=float)

x = np.arange(len(labels))
bar_width = 0.34

# =====================
# Plot
# =====================
fig, ax1 = plt.subplots(figsize=(8.5, 4.8))

# Bars: number of QUIC-enabled addresses
bars_ipv4 = ax1.bar(
    x - bar_width / 2,
    ipv4_count,
    width=bar_width,
    label='QUIC-enabled IPv4 addresses',
    edgecolor='black',
    linewidth=0.6
)

bars_ipv6 = ax1.bar(
    x + bar_width / 2,
    ipv6_count,
    width=bar_width,
    label='QUIC-enabled IPv6 addresses (hitlist)',
    edgecolor='black',
    linewidth=0.6,
    hatch='//'
)

ax1.set_ylabel('Number of QUIC-enabled addresses')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{int(v):,}'))
ax1.grid(axis='y', linestyle='--', alpha=0.35)

# Line: percentage
ax2 = ax1.twinx()

line_ipv4, = ax2.plot(
    x,
    ipv4_percent,
    marker='o',
    linewidth=2,
    label='IPv4 QUIC-enabled ratio'
)

line_ipv6, = ax2.plot(
    x,
    ipv6_percent,
    marker='s',
    linewidth=2,
    linestyle='--',
    label='IPv6 QUIC-enabled ratio (hitlist)'
)

ax2.set_ylabel('Percentage of total targets (%)')
ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.3f}%'))

# =====================
# Labels on bars
# =====================
def add_bar_labels(ax, bars):
    for bar in bars:
        height = bar.get_height()
        if np.isnan(height):
            continue
        ax.annotate(
            f'{int(height):,}',
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=8
        )

add_bar_labels(ax1, bars_ipv4)
add_bar_labels(ax1, bars_ipv6)

# =====================
# Labels on line points
# =====================
def add_line_labels(ax, xs, ys, dy=6):
    for xi, yi in zip(xs, ys):
        if np.isnan(yi):
            continue
        ax.annotate(
            f'{yi:.3f}%',
            xy=(xi, yi),
            xytext=(0, dy),
            textcoords='offset points',
            ha='center',
            va='bottom',
            fontsize=8
        )

add_line_labels(ax2, x, ipv4_percent, dy=6)
add_line_labels(ax2, x, ipv6_percent, dy=6)

# =====================
# Legend
# =====================
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc='upper left',
    fontsize=8,
    frameon=True
)

# =====================
# Title and layout
# =====================
ax1.set_title('Growth of QUIC Adoption in Vietnam: IPv4 and IPv6')

# Optional note inside figure
ax1.text(
    0.01,
    0.94,
    'IPv4 total: 16,486,192 addresses\nIPv6 total: 856,000 hitlist targets',
    transform=ax1.transAxes,
    fontsize=8,
    va='top',
    bbox=dict(boxstyle='round,pad=0.35', alpha=0.15)
)

fig.tight_layout()

# Save
plt.savefig('fig_quic_growth_ipv4_ipv6_dual_axis.pdf', bbox_inches='tight')
plt.savefig('fig_quic_growth_ipv4_ipv6_dual_axis.png', dpi=300, bbox_inches='tight')

plt.show()